# Vector stores and semantic search



In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [2]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata
        self.embedding = None


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document
        self.document_embeddings: list[np.ndarray] = []

class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
      self.embedding_model = embedding_model
      self.documents = []
      self.document_embeddings = []

    def add_documents(self, documents):
      texts = [doc.text for doc in documents]
      embeddings = self.embedding_model.encode(texts)

      for doc, emb in zip(documents, embeddings):
          doc.embedding = emb
          self.documents.append(doc)
          self.document_embeddings.append(emb)

    def search(self, query: str, top_k: int = 5) -> list:
      if not self.documents:
          return []

      query_embedding = self.embedding_model.encode([query])

      similarities = cosine_similarity(query_embedding, self.document_embeddings)[0]

      top_indices = np.argsort(similarities)[::-1][:top_k]

      results = []
      for idx in top_indices:
          results.append(SearchResult(similarities[idx], self.documents[idx]))

      return results

In [3]:
fun_facts_doc = pd.read_csv('animal-fun-facts-dataset.csv')
fun_facts_doc.head()

,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark


In [11]:
# Carga de datos
documents = []
for _, row in fun_facts_doc.iterrows():
    if pd.notna(row['text']):
        metadata = {
            'animal_name': row['animal_name'],
            'source': row['source'] if pd.notna(row['source']) else '',
            'media_link': row['media_link'] if pd.notna(row['media_link']) else '',
            'wikipedia_link': row['wikipedia_link'] if pd.notna(row['wikipedia_link']) else ''
        }
        documents.append(Document(row['text'], metadata))

model = SentenceTransformer('all-MiniLM-L6-v2')
vector_store = VectorStore(model)
vector_store.add_documents(documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# Ejemplos

searches = [
  "biggest weight",
  "longest sleep time",
  "animals with long limbs",
  "animals that jump the highest",
  "slowest birthrate of a species"
]

# Busquedas
for search in searches:
  results = vector_store.search(search, top_k=3)
  print(f"SEARCH: {search}")
  for r in results:
    print(f"Score: {r.score:.4f} - {r.document.metadata['animal_name']}: {r.document.text[:80]}...")
    print("Metadata: ", r.document.metadata)
    print()
  print()


SEARCH: biggest weight
Score: 0.5675 - great bustard: The great bustard is the heaviest bird in the world capable of flight. It can we...
Metadata:  {'animal_name': 'great bustard', 'source': 'https://reddit.com/r/Awwducational/comments/yfz0q6/the_great_bustard_is_the_heaviest_bird_in_the/', 'media_link': 'https://i.redd.it/k5ooikby8nw91.png', 'wikipedia_link': '/wiki/Great_bustard'}

Score: 0.5498 - grant's gazelle: Adult males are seen as the largest gazelle concerning body weight....
Metadata:  {'animal_name': "grant's gazelle", 'source': 'https://seaworld.org/animals/facts/mammals/grants-gazelle/', 'media_link': '', 'wikipedia_link': '/wiki/Grant%27s_gazelle'}

Score: 0.5331 - sperm whale: They are the largest of the toothed whales.
Adult males can weigh up to 80 tonne...
Metadata:  {'animal_name': 'sperm whale', 'source': 'https://factanimal.com/sperm-whale/', 'media_link': '', 'wikipedia_link': '/wiki/Sperm_whale'}


SEARCH: longest sleep time
Score: 0.5063 - koala: A koala spend

## Part II: Filtering by metadata

In [15]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass